# Aula 16 — Problemas eulerianos e inspeção da infraestrutura

**ECAA08 · Grupo 06 · Drone agrícola de pulverização**


## Modelo da inspeção

O objetivo é percorrer cada trecho de acesso de manutenção uma vez. Usa-se um **multigrafo não dirigido de acesso físico**, independente da orientação de escoamento da calda. O percurso representa um técnico ou veículo de inspeção na estação, com o drone em manutenção; não pressupõe um robô dentro das mangueiras nem voo de pulverização.

Seis locais são conectados por um anel: BASE, TANQUE, BOMBA_SOLO, ABASTECIMENTO, BANCADA_DRONE e LAVAGEM. Acrescentam-se três acessos entre TANQUE, BOMBA_SOLO e ABASTECIMENTO. Dois pares de vértices passam a possuir arestas paralelas, cada qual com ID único.


## Critérios de Euler

Um multigrafo não dirigido admite circuito euleriano se todos os vértices com grau não nulo pertencem a uma mesma componente e todos os graus são pares. Para uma trilha aberta, são necessários exatamente dois vértices ímpares e a partida deve ocorrer em um deles. Vértices isolados sem arestas não afetam a cobertura dos trechos.

No exemplo de nove arestas, os graus são 2, 4, 4, 4, 2, 2. A soma é 18, igual a $2|E|$. Graus pares sozinhos não bastam: dois triângulos desconectados têm todos os graus pares e não admitem um circuito que percorra ambos.


## Hierholzer

O algoritmo mantém uma pilha, consome arestas por ID e retrocede quando não há mais trecho disponível. A inversão da sequência de retrocesso produz a trilha. Com listas de adjacência e conjunto de IDs consumidos, cada incidência é examinada uma quantidade constante de vezes: $O(V+E)$.

O resultado contém a sequência de vértices e a sequência de IDs. Verifica-se que existem dez posições de vértices e nove IDs, cada ID aparece uma vez e cada trecho une exatamente os vértices adjacentes indicados. A verificação por pares de nomes seria insuficiente nas arestas paralelas.


## Exemplo complementar

Remover o último acesso adicional deixa dois vértices ímpares. O circuito fechado deixa de existir, mas há trilha aberta partindo de um deles. Se a tarefa exigisse retorno à base, seria necessário repetir trechos; minimizar essa repetição conduz ao problema do carteiro chinês, que não é resolvido aqui.

Hierholzer constrói uma cobertura sem repetição quando ela existe. Não escolhe o menor percurso entre soluções com tempos de serviço, sentidos proibidos ou repetições obrigatórias. No circuito euleriano, cada aresta é usada uma vez, logo a soma dos pesos de todas as arestas seria a mesma em qualquer circuito.

**Entregável:** implementação que trata paralelas corretamente, verifica conectividade e paridade, gera circuito/trilha e rejeita o caso desconectado.


## Implementação executável

Python 3.10 ou superior. Execute de cima para baixo. As definições estão incluídas para permitir abrir apenas este notebook no Jupyter ou Colab. Nenhuma conexão com equipamento real é realizada.

In [1]:
from collections import deque, Counter
from math import inf, isfinite, hypot
from heapq import heappush, heappop
from itertools import permutations
from copy import deepcopy


In [2]:
def tabela(linhas):
    if not linhas:
        return '(sem registros)'
    cols = list(linhas[0])
    larg = {c: max(len(c), *(len(str(r.get(c, ''))) for r in linhas)) for c in cols}
    fmt = lambda r: ' | '.join(str(r.get(c, '')).ljust(larg[c]) for c in cols)
    return '\n'.join([fmt(dict(zip(cols, cols))), '-+-'.join('-'*larg[c] for c in cols)] + [fmt(r) for r in linhas])


In [3]:
def hierholzer(vertices, arestas, inicio):
    """Multigrafo não dirigido; arestas=(ID,u,v). Retorna vértices e IDs."""
    adj = {v: [] for v in vertices}
    ids = [id_ for id_,_,_ in arestas]
    if len(set(ids)) != len(ids): raise ValueError('IDs repetidos')
    if inicio not in adj: raise KeyError(inicio)
    for id_,u,v in arestas:
        if u not in adj or v not in adj: raise KeyError('Extremidade ausente')
        adj[u].append((v,id_)); adj[v].append((u,id_))
    ativos = {v for v in adj if adj[v]}
    vistos, pilha = set(), [inicio]
    while pilha:
        u = pilha.pop()
        if u in vistos: continue
        vistos.add(u); pilha.extend(v for v,_ in adj[u] if v not in vistos)
    if not ativos <= vistos: raise ValueError('Componentes com arestas desconectadas')
    impares = {v for v in adj if len(adj[v]) % 2}
    if len(impares) not in (0,2) or (impares and inicio not in impares):
        raise ValueError('Graus/início incompatíveis com trilha euleriana')
    usados, pilha, saida = set(), [(inicio,None)], []
    while pilha:
        u = pilha[-1][0]
        while adj[u] and adj[u][-1][1] in usados: adj[u].pop()
        if adj[u]:
            v,id_ = adj[u].pop(); usados.add(id_); pilha.append((v,id_))
        else: saida.append(pilha.pop())
    saida.reverse()
    rota = [v for v,_ in saida]
    percurso = [id_ for _,id_ in saida[1:]]
    if len(usados) != len(arestas): raise ValueError('Nem todas as arestas visitadas')
    return rota, percurso


## Experimento 1 — Circuito de inspeção e validação por ID

In [4]:
locais=['BASE','TANQUE','BOMBA_SOLO','ABASTECIMENTO','BANCADA_DRONE','LAVAGEM']
arestas=[('i1',locais[0],locais[1]),('i2',locais[1],locais[2]),
         ('i3',locais[2],locais[3]),('i4',locais[3],locais[4]),
         ('i5',locais[4],locais[5]),('i6',locais[5],locais[0]),
         ('i7',locais[1],locais[3]),('i8',locais[3],locais[2]),('i9',locais[2],locais[1])]
graus=Counter(v for _,u,w in arestas for v in (u,w))
rota,ids=hierholzer(locais,arestas,'BASE')
print('Graus:',dict(graus))
print('Circuito:', ' -> '.join(rota))
print('IDs:',ids)
assert sum(graus.values())==18 and all(g%2==0 for g in graus.values())
assert len(rota)==10 and rota[0]==rota[-1]=='BASE'
assert Counter(ids)==Counter(id_ for id_,_,_ in arestas)
mapa={id_:{u,v} for id_,u,v in arestas}
assert all({u,v}==mapa[id_] for u,v,id_ in zip(rota,rota[1:],ids))


Graus: {'BASE': 2, 'TANQUE': 4, 'BOMBA_SOLO': 4, 'ABASTECIMENTO': 4, 'BANCADA_DRONE': 2, 'LAVAGEM': 2}
Circuito: BASE -> LAVAGEM -> BANCADA_DRONE -> ABASTECIMENTO -> BOMBA_SOLO -> TANQUE -> ABASTECIMENTO -> BOMBA_SOLO -> TANQUE -> BASE
IDs: ['i6', 'i5', 'i4', 'i8', 'i9', 'i7', 'i3', 'i2', 'i1']


## Experimento 2 — Trilha aberta e contraexemplo desconectado

In [5]:
aberta,ids_abertos=hierholzer(locais,arestas[:-1],'TANQUE')
assert aberta[0]=='TANQUE' and aberta[-1]=='BOMBA_SOLO'
assert len(ids_abertos)==8
print('Trilha aberta:',aberta)
dois_triangulos=[('a','A','B'),('b','B','C'),('c','C','A'),
                 ('d','D','E'),('e','E','F'),('f','F','D')]
try:
    hierholzer(list('ABCDEF'),dois_triangulos,'A')
except ValueError as exc: print('Rejeição correta:',exc)
else: raise AssertionError('Paridade sem conectividade não basta')


Trilha aberta: ['TANQUE', 'ABASTECIMENTO', 'BOMBA_SOLO', 'ABASTECIMENTO', 'BANCADA_DRONE', 'LAVAGEM', 'BASE', 'TANQUE', 'BOMBA_SOLO']
Rejeição correta: Componentes com arestas desconectadas
